# Chapter 38: Databases with SQLAlchemy — Colab Notebook

This notebook actually runs the SQLAlchemy + FastAPI code that the lesson page shows as
reference-only. Every cell below is meant to be executed for real, in order, using an
in-memory SQLite database (`sqlite+aiosqlite:///:memory:`) and FastAPI's `TestClient` —
no real socket, port, or file needed.

First, install the dependencies this notebook needs (only required once per Colab session):

```
!pip install -q fastapi uvicorn httpx sqlalchemy aiosqlite pydantic
```


In [1]:
!pip install -q fastapi uvicorn httpx sqlalchemy aiosqlite pydantic
import fastapi, sqlalchemy
print('fastapi', fastapi.__version__)
print('sqlalchemy', sqlalchemy.__version__)


fastapi 0.141.1
sqlalchemy 2.0.54


## 38.1 — Why A Real Database

A plain Python dict loses everything on restart and has no safe concurrent-write story. This notebook replaces it with a real (in-memory) SQL database via SQLAlchemy's async ORM.

In [2]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

print('Base declared:', Base)


Base declared: <class '__main__.Base'>


## 38.2 — Defining a SQLAlchemy Model

In [3]:
class Candidate(Base):
    __tablename__ = "candidates"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    city: Mapped[str]
    score: Mapped[float]

print('Candidate table:', Candidate.__tablename__)
print('Columns:', [c.name for c in Candidate.__table__.columns])


Candidate table: candidates
Columns: ['id', 'name', 'city', 'score']


## 38.3 — Async Engine & Session

In [4]:
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession

engine = create_async_engine("sqlite+aiosqlite:///:memory:")
SessionLocal = async_sessionmaker(engine, expire_on_commit=False)

async def init_db():
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

await init_db()
print('Tables created.')


Tables created.


## 38.4 — Depends() for a DB Session

In [5]:
from typing import AsyncGenerator
from fastapi import FastAPI, Depends, HTTPException
from pydantic import BaseModel, ConfigDict, Field
from sqlalchemy import select

app = FastAPI(title="Candidate Scoring API v2")

async def get_session() -> AsyncGenerator[AsyncSession, None]:
    async with SessionLocal() as session:
        yield session

print('get_session defined.')


get_session defined.


## 38.5 — Create (POST)

In [6]:
class CandidateIn(BaseModel):
    name: str
    city: str
    score: float = Field(gt=0, le=1)

class CandidateOut(BaseModel):
    id: int
    name: str
    city: str
    score: float
    model_config = ConfigDict(from_attributes=True)

@app.post("/candidates", response_model=CandidateOut, status_code=201)
async def create_candidate(candidate: CandidateIn, session: AsyncSession = Depends(get_session)):
    row = Candidate(name=candidate.name, city=candidate.city, score=candidate.score)
    session.add(row)
    await session.commit()
    await session.refresh(row)
    return row

print('POST /candidates defined.')


POST /candidates defined.


## 38.6 — Read (GET)

In [7]:
@app.get("/candidates/{candidate_id}", response_model=CandidateOut)
async def get_candidate(candidate_id: int, session: AsyncSession = Depends(get_session)):
    row = await session.get(Candidate, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Candidate not found")
    return row

@app.get("/candidates", response_model=list[CandidateOut])
async def list_candidates(min_score: float = 0.0, session: AsyncSession = Depends(get_session)):
    result = await session.execute(select(Candidate).where(Candidate.score >= min_score))
    return result.scalars().all()

print('GET routes defined.')


GET routes defined.


## 38.7 — Update & Delete

In [8]:
@app.delete("/candidates/{candidate_id}", status_code=204)
async def delete_candidate(candidate_id: int, session: AsyncSession = Depends(get_session)):
    row = await session.get(Candidate, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Candidate not found")
    await session.delete(row)
    await session.commit()

print('DELETE route defined.')


DELETE route defined.


## 38.8 — Testing DB-Backed Endpoints, End to End

In [9]:
from fastapi.testclient import TestClient

client = TestClient(app)

r1 = client.post("/candidates", json={"name": "Alice", "city": "Hubli", "score": 0.92})
print('POST status:', r1.status_code)
print('POST body:', r1.json())
assert r1.status_code == 201
created_id = r1.json()["id"]

r2 = client.get(f"/candidates/{created_id}")
print('GET by id status:', r2.status_code, r2.json())
assert r2.status_code == 200
assert r2.json()["name"] == "Alice"

client.post("/candidates", json={"name": "Bob", "city": "Pune", "score": 0.4})
r3 = client.get("/candidates", params={"min_score": 0.8})
print('GET filtered:', r3.json())
assert all(c["score"] >= 0.8 for c in r3.json())

r4 = client.delete(f"/candidates/{created_id}")
print('DELETE status:', r4.status_code)
assert r4.status_code == 204

r5 = client.get(f"/candidates/{created_id}")
print('GET after delete status:', r5.status_code)
assert r5.status_code == 404

print('All database-backed assertions passed.')


POST status: 201
POST body: {'id': 1, 'name': 'Alice', 'city': 'Hubli', 'score': 0.92}
GET by id status: 200 {'id': 1, 'name': 'Alice', 'city': 'Hubli', 'score': 0.92}
GET filtered: [{'id': 1, 'name': 'Alice', 'city': 'Hubli', 'score': 0.92}]
DELETE status: 204
GET after delete status: 404
All database-backed assertions passed.


## Mini Project: Candidate Scoring API v2 — Real Database

A fresh app, database, and TestClient run through create/read/filter/delete end to end.

In [10]:
project_engine = create_async_engine("sqlite+aiosqlite:///:memory:")
ProjectSessionLocal = async_sessionmaker(project_engine, expire_on_commit=False)

async def project_get_session():
    async with ProjectSessionLocal() as session:
        yield session

project_app = FastAPI(title="Candidate Scoring API v2 - Project")
project_app.dependency_overrides = {}

async def project_init_db():
    async with project_engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)

await project_init_db()

@project_app.post("/candidates", response_model=CandidateOut, status_code=201)
async def p_create(candidate: CandidateIn, session: AsyncSession = Depends(project_get_session)):
    row = Candidate(name=candidate.name, city=candidate.city, score=candidate.score)
    session.add(row); await session.commit(); await session.refresh(row)
    return row

@project_app.get("/candidates/{candidate_id}", response_model=CandidateOut)
async def p_get(candidate_id: int, session: AsyncSession = Depends(project_get_session)):
    row = await session.get(Candidate, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Candidate not found")
    return row

@project_app.get("/candidates", response_model=list[CandidateOut])
async def p_list(min_score: float = 0.0, session: AsyncSession = Depends(project_get_session)):
    result = await session.execute(select(Candidate).where(Candidate.score >= min_score))
    return result.scalars().all()

@project_app.delete("/candidates/{candidate_id}", status_code=204)
async def p_delete(candidate_id: int, session: AsyncSession = Depends(project_get_session)):
    row = await session.get(Candidate, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail="Candidate not found")
    await session.delete(row); await session.commit()

pclient = TestClient(project_app)

resp = pclient.post("/candidates", json={"name": "Carol", "city": "Mumbai", "score": 0.81})
assert resp.status_code == 201
cid = resp.json()["id"]

assert pclient.get(f"/candidates/{cid}").status_code == 200
pclient.post("/candidates", json={"name": "Dave", "city": "Delhi", "score": 0.2})
filtered = pclient.get("/candidates", params={"min_score": 0.5}).json()
assert all(c["score"] >= 0.5 for c in filtered)
assert pclient.delete(f"/candidates/{cid}").status_code == 204
assert pclient.get(f"/candidates/{cid}").status_code == 404

print('Project checklist: PASSED (create, get, filter, delete all verified against a real in-memory DB)')


Project checklist: PASSED (create, get, filter, delete all verified against a real in-memory DB)


### Next: Chapter 39 — Authentication & Security (Colab)

Protecting these endpoints with OAuth2PasswordBearer, JWT tokens, and real password hashing.